# Federated Random Forest via Union-of-Forests

**One of three method-focused notebooks for the BioITWorld / ISMB talk.**

This notebook does exactly one thing: fit a federated random-forest
regressor to predict log(C-peptide AUC), without pooling raw data across
institutions. Unlike the linear methods, there are no coefficients to
average — so the federation rule is different.

## The Union-of-Forests aggregation

A random forest is already an ensemble: many decision trees, each fit on
a bootstrap resample, predictions averaged. The federated extension is
the natural one:

> Each institution grows its **own** random forest on its **own** data.
> The federated model is the **union of all institutions' trees**. A new
> subject's prediction is the average prediction across every tree from
> every institution.

This is sometimes called **Union-of-Forests** or a "forest of federated
forests." It has three properties that make it attractive for our setting:

1. **No raw data crosses the wire.** Each institution ships only its
   trained trees (the split thresholds and leaf values), never subject
   records.
2. **No iterative communication.** Unlike ADMM, there's a single round:
   train locally, ship the forest, done. Trees from different institutions
   never need to agree with each other.
3. **It captures non-linearity.** Trees model interactions and thresholds
   that the linear methods (MLR, LASSO) cannot. This is the whole reason
   to include a forest in the comparison.

The cost: a forest cannot extrapolate beyond the feature ranges it saw
during training, and a tiny institution's trees may be noisy. The
sample-size imbalance across our studies (SDY524 has 75 subjects, SDY569
has 10) means SDY569's trees contribute equally per-tree but the forest
has fewer of them — a natural down-weighting.

## Data panels in this notebook

- **Panel A** — 5 autoantibodies + 3 age-group dummies + Sex, N=150
  across 4 studies.
- **Panel B** — Jeff-extended 15 features, N=98 across 3 studies.


## 1. Setup

In [ ]:
from __future__ import annotations
import sys, os, warnings
from pathlib import Path
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

REPO = Path.cwd()
if (REPO / "src").exists():
    sys.path.insert(0, str(REPO / "src"))
elif (REPO.parent / "src").exists():
    REPO = REPO.parent
    sys.path.insert(0, str(REPO / "src"))
os.chdir(REPO)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error
from scipy.stats import f as fdist, ncf

import oadr_data as od

RNG_SEED = 42
N_FOLDS = 5
ALPHA_STAT = 0.05
N_TREES = 200
np.random.seed(RNG_SEED)

(REPO / "results").mkdir(exist_ok=True)
(REPO / "figures").mkdir(exist_ok=True)
print(f"Repo root: {REPO}")


### A note on `src/oadr_data.py`

The single shared utility in this project, holding the per-study quirk
normalisation. Everything else in this notebook is inline.


In [ ]:
A = od.load_panel_a_all()
Xa = A[od.PANEL_A_FEATURES].values.astype(float)
ya = A[od.PANEL_A_TARGET].values.astype(float)
studies_a = A["Study"].values
fa = list(od.PANEL_A_FEATURES)

B = od.load_panel_b_all()
Xb_df, yb_s, fb = od.panel_b_design_matrix(B)
Xb = Xb_df.values.astype(float)
yb = yb_s.values.astype(float)
studies_b = B["Study"].values
fb = list(fb)

print(f"Panel A: N={len(A)}, features={len(fa)}")
print(A.groupby("Study").size().to_string())
print()
print(f"Panel B: N={len(B)}, features={len(fb)}")
print(B.groupby("Study").size().to_string())


## 2. Power calculation utility

The same helpers used in every notebook. For a random forest, "number of
predictors k" in the power calculation is the nominal feature count — a
forest doesn't shrink features the way LASSO does, so we report the full
panel width. Power from the non-central F distribution directly.


In [ ]:
def calc_power(n, k, f2, alpha=ALPHA_STAT):
    if n <= k + 1 or f2 <= 0:
        return float("nan")
    df_num, df_denom = k, n - k - 1
    F_crit = fdist.ppf(1 - alpha, df_num, df_denom)
    nc = f2 * n
    return float(1 - ncf.cdf(F_crit, df_num, df_denom, nc))


def f2_from_r2(r2):
    return r2 / (1 - r2) if 0 < r2 < 1 else 0.0


def metrics(y_true, y_pred):
    mask = ~np.isnan(y_pred)
    if mask.sum() < 2:
        return float("nan"), float("nan"), int(mask.sum())
    mse = float(np.mean((y_true[mask] - y_pred[mask]) ** 2))
    rss = float(np.sum((y_true[mask] - y_pred[mask]) ** 2))
    tss = float(np.sum((y_true[mask] - y_true[mask].mean()) ** 2))
    r2 = 1 - rss / tss if tss > 0 else float("nan")
    return mse, r2, int(mask.sum())


## 3. The Union-of-Forests algorithm, written out long-hand

Two loops:

1. **Cross-validation outer loop**: 5 folds, stratified by `Study`.
2. **Per-institution inner loop**: each institution fits its own
   `RandomForestRegressor` on its own training subjects, with a
   per-institution `MinMaxScaler`. The trained forest is kept locally.

At prediction time, each held-out subject is scaled with **their own
institution's scaler**, then run through **every institution's forest**.
The final prediction is the mean across all institutions' forest
predictions — the Union-of-Forests.

> Note: averaging the *forest-level* predictions (one prediction per
> institution, then averaged) weights each institution equally. An
> alternative would average across all *trees* pooled together, which
> weights institutions by how many trees they contribute. We use
> forest-level averaging here so a large institution doesn't dominate
> purely by growing more trees; all our forests use the same `N_TREES`.


In [ ]:
def federated_rf_oof(X, y, studies, n_splits=N_FOLDS, n_trees=N_TREES, seed=RNG_SEED):
    """Out-of-fold predictions under federated random forest (Union-of-Forests).

    Returns an array `oof`; oof[i] is the prediction for subject i, averaged
    across every institution's forest, from a fold where subject i was held out.
    """
    X = np.asarray(X); y = np.asarray(y)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof = np.full_like(y, np.nan, dtype=float)

    for tr, te in skf.split(X, studies):
        # 1. Per-institution local forests
        local_forests = {}   # study_id -> fitted RandomForestRegressor
        scalers = {}         # study_id -> MinMaxScaler fit on that institution's training data
        for s in sorted(set(studies[tr])):
            idx = tr[studies[tr] == s]
            if len(idx) < 2:
                continue
            sc = MinMaxScaler().fit(X[idx])
            scalers[s] = sc
            rf = RandomForestRegressor(n_estimators=n_trees, min_samples_leaf=2,
                                       n_jobs=1, random_state=seed)
            rf.fit(sc.transform(X[idx]), y[idx])
            local_forests[s] = rf

        # 2. Predict each held-out subject through every institution's forest, average
        for i in te:
            preds = []
            for s_k, rf in local_forests.items():
                xs = scalers[s_k].transform(X[i:i+1])
                preds.append(rf.predict(xs)[0])
            if preds:
                oof[i] = np.mean(preds)

    return oof


## 4. Run it on Panel A and Panel B

The same four configurations used in the other method notebooks, so the
results line up row-for-row when assembled into the cross-method summary.


In [ ]:
configs = [
    ("Panel A, all 4 studies", Xa, ya, studies_a, fa),
    ("Panel A, drop SDY1737",  Xa[studies_a != "SDY1737"],
                              ya[studies_a != "SDY1737"],
                              studies_a[studies_a != "SDY1737"], fa),
    ("Panel B, all 3 studies", Xb, yb, studies_b, fb),
    ("Panel B, drop SDY1737",  Xb[studies_b != "SDY1737"],
                              yb[studies_b != "SDY1737"],
                              studies_b[studies_b != "SDY1737"], fb),
]

rows = []
for name, X, y, st, fnames in configs:
    oof = federated_rf_oof(X, y, st)
    mse, r2, n = metrics(y, oof)
    f2 = f2_from_r2(r2)
    power = calc_power(n, len(fnames), f2)
    rows.append({"config": name, "N": n, "k": len(fnames),
                 "MSE": mse, "R2": r2, "Cohen_f2": f2,
                 "achieved_power": power})
    print(f"{name:30s}  N={n:3d}  MSE={mse:.3f}  R²={r2:+.3f}  f²={f2:.3f}  power={power:.3f}")

rf_results = pd.DataFrame(rows)
rf_results.to_csv("results/federated_rf_results.csv", index=False)


## 5. SDY569-alone vs federated scatter

The same two-panel scatter as the other methods. Left: SDY569 predicted
by its own forest alone (10 subjects). Right: every subject predicted by
the Union-of-Forests across all institutions.


In [ ]:
def rf_solo_vs_federated_oof(X, y, studies, n_splits=N_FOLDS, n_trees=N_TREES, seed=RNG_SEED):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof_solo = np.full_like(y, np.nan, dtype=float)
    oof_fed = np.full_like(y, np.nan, dtype=float)

    for tr, te in skf.split(X, studies):
        # SDY569 alone
        tr569 = tr[studies[tr] == "SDY569"]
        te569 = te[studies[te] == "SDY569"]
        if len(tr569) >= 2 and len(te569) >= 1:
            sc = MinMaxScaler().fit(X[tr569])
            rf = RandomForestRegressor(n_estimators=n_trees, min_samples_leaf=2,
                                       n_jobs=1, random_state=seed)
            rf.fit(sc.transform(X[tr569]), y[tr569])
            oof_solo[te569] = rf.predict(sc.transform(X[te569]))

        # Union-of-Forests across all institutions
        local_forests, scalers = {}, {}
        for s in sorted(set(studies[tr])):
            idx = tr[studies[tr] == s]
            if len(idx) < 2:
                continue
            sc = MinMaxScaler().fit(X[idx])
            scalers[s] = sc
            rf = RandomForestRegressor(n_estimators=n_trees, min_samples_leaf=2,
                                       n_jobs=1, random_state=seed)
            rf.fit(sc.transform(X[idx]), y[idx])
            local_forests[s] = rf
        for i in te:
            preds = []
            for s_k, rf in local_forests.items():
                xs = scalers[s_k].transform(X[i:i+1])
                preds.append(rf.predict(xs)[0])
            if preds:
                oof_fed[i] = np.mean(preds)

    return oof_solo, oof_fed


oof_solo, oof_fed = rf_solo_vs_federated_oof(Xa, ya, studies_a)


In [ ]:
study_colors = {"SDY524": "#1f77b4", "SDY569": "#d62728",
                "SDY797": "#2ca02c", "SDY1737": "#9467bd"}

is569 = (studies_a == "SDY569")
mse_solo, r2_solo, n_solo = metrics(ya[is569], oof_solo[is569])
mse_fed, r2_fed, n_fed = metrics(ya, oof_fed)

fig, axes = plt.subplots(1, 2, figsize=(11.5, 5), constrained_layout=True)

ax = axes[0]
mask = is569 & ~np.isnan(oof_solo)
ax.scatter(ya[mask], oof_solo[mask], c=study_colors["SDY569"], s=65,
           edgecolor="white", linewidth=0.6, label=f"SDY569 (N={mask.sum()})")
if mask.sum() > 0:
    lo = min(ya[mask].min(), oof_solo[mask].min())
    hi = max(ya[mask].max(), oof_solo[mask].max())
    ax.plot([lo, hi], [lo, hi], "k--", alpha=0.4, label="y = x")
ax.set_xlabel("Observed log(C-peptide AUC)")
ax.set_ylabel("Predicted log(C-peptide AUC)")
ax.set_title(f"SDY569 alone  (N = {mask.sum()})\nMSE = {mse_solo:.3f}    R² = {r2_solo:+.3f}",
             fontweight="bold")
ax.legend(loc="lower right", fontsize=9); ax.grid(alpha=0.25)

ax = axes[1]
for s in sorted(set(studies_a)):
    mask = (studies_a == s) & ~np.isnan(oof_fed)
    if mask.sum() == 0:
        continue
    ax.scatter(ya[mask], oof_fed[mask], c=study_colors.get(s, "grey"),
               s=65, edgecolor="white", linewidth=0.6, alpha=0.85,
               label=f"{s} (N={mask.sum()})")
all_mask = ~np.isnan(oof_fed)
if all_mask.sum() > 0:
    lo = min(ya[all_mask].min(), oof_fed[all_mask].min())
    hi = max(ya[all_mask].max(), oof_fed[all_mask].max())
    ax.plot([lo, hi], [lo, hi], "k--", alpha=0.4, label="y = x")
ax.set_xlabel("Observed log(C-peptide AUC)")
ax.set_ylabel("Predicted log(C-peptide AUC)")
ax.set_title(f"Union-of-Forests  (N = {all_mask.sum()})\nMSE = {mse_fed:.3f}    R² = {r2_fed:+.3f}",
             fontweight="bold")
ax.legend(loc="lower right", fontsize=9); ax.grid(alpha=0.25)

fig.suptitle("Federated Random Forest — SDY569 alone vs Union-of-Forests (Panel A)",
             fontsize=13, fontweight="bold")
fig.savefig("figures/federated_rf_solo_vs_fed.pdf", dpi=300)
fig.savefig("figures/federated_rf_solo_vs_fed.png", dpi=220)
plt.show()


## 6. Feature importance per institution

A forest reports which features it relied on. Because each institution
grows its own forest, we can compare *which features each institution
found important* — a federated view that pooling would obscure. The bars
below show the mean impurity-decrease importance per feature, one grouped
bar per institution.


In [ ]:
# Fit one forest per institution on its full data, collect feature importances
importances = {}
for s in sorted(set(studies_a)):
    idx = (studies_a == s).nonzero()[0]
    sc = MinMaxScaler().fit(Xa[idx])
    rf = RandomForestRegressor(n_estimators=N_TREES, min_samples_leaf=2,
                               n_jobs=1, random_state=RNG_SEED)
    rf.fit(sc.transform(Xa[idx]), ya[idx])
    importances[s] = rf.feature_importances_

imp_df = pd.DataFrame(importances, index=fa)
imp_df.to_csv("results/federated_rf_feature_importance.csv")
print("Per-institution feature importances (Panel A):")
print(imp_df.round(3).to_string())

fig, ax = plt.subplots(figsize=(10, 5))
n_studies = len(importances)
width = 0.8 / n_studies
x = np.arange(len(fa))
for j, s in enumerate(sorted(importances)):
    ax.bar(x + j * width, importances[s], width, label=s,
           color=study_colors.get(s, "grey"))
ax.set_xticks(x + width * (n_studies - 1) / 2)
ax.set_xticklabels(fa, rotation=45, ha="right")
ax.set_ylabel("Impurity-decrease importance")
ax.set_title("Random-forest feature importance by institution (Panel A)")
ax.legend(title="Institution")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig("figures/federated_rf_feature_importance.pdf", dpi=300)
fig.savefig("figures/federated_rf_feature_importance.png", dpi=220)
plt.show()


## 7. Talk-ready summary

One row per configuration: N, k, MSE, R², achieved power. The numbers
line up with the MLR and LASSO notebooks so the three can be assembled
into a single cross-method comparison slide.


In [ ]:
print("Federated Random Forest (Union-of-Forests) — talk summary")
print("=" * 78)
print(rf_results.to_string(index=False))
print()
print("Saved to results/federated_rf_results.csv")


## 8. What this notebook does *not* do

- **No pooled comparison.** Every forest is grown on a single
  institution's subjects. There is no `RandomForest.fit(Xa, ya)` on the
  union of subject-level data.
- **No linear methods.** MLR is in `Federated_MLR.ipynb`, LASSO in
  `Federated_LASSO.ipynb`.
- **No CNN.** The deep model has its own notebook.

The random forest is included here because it's the **non-linear**
counterpoint to the linear methods: if a forest substantially beats
MLR/LASSO on the same features, that's evidence of interactions or
thresholds the linear models can't represent — the same argument the
autoencoder-CNN makes, but with an interpretable model.

## Suggested next reading

- `Federated_MLR.ipynb` and `Federated_LASSO.ipynb` — the linear methods
  this forest is compared against.
